# HackUDC 2026 — Inditex Fashion Retrieval (Google Colab Version)
**Task:** Given a bundle (model photo with multiple garments), retrieve up to 15 matching
product IDs from a 27K-product catalog.
**Metric:** Recall@15

## Cell 1: Setup & Google Drive Mount

In [ ]:
from google.colab import drive
import subprocess, sys

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# Install dependencies
_pip(
    "fashion-clip", "transformers>=4.35.0", "accelerate",
    "tqdm", "Pillow", "requests", "open_clip_torch",
    "pandas", "numpy", "matplotlib"
)

# faiss: try GPU build first, fall back to CPU
try:
    _pip("faiss-gpu")
except Exception:
    try:
        _pip("faiss-cpu")
    except Exception:
        pass

print("✓ All packages installed.")

## Cell 2: Imports & Configuration

In [ ]:
import os
import gc
import json
import math
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
import torch
try:
    import faiss
except ModuleNotFoundError:
    raise ModuleNotFoundError("faiss not found. Re-run Cell 1.")
from PIL import Image
from tqdm.auto import tqdm

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

warnings.filterwarnings("ignore")

# ── Paths (Google Drive) ───────────────────────────────────────────────────────
BASE_DIR  = Path("/content/drive/MyDrive/GCED/hackudc")
DATA_DIR  = BASE_DIR / "data"
WORK_DIR  = BASE_DIR / "working"
IMG_DIR   = WORK_DIR / "images"
PROD_DIR  = IMG_DIR / "products"
BUND_DIR  = IMG_DIR / "bundles"
SUBM_FILE = WORK_DIR / "submission.csv"

for d in [PROD_DIR, BUND_DIR, WORK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Base dir: {BASE_DIR}")
print(f"Data dir: {DATA_DIR}")
print(f"Work dir: {WORK_DIR}")

# ── Hardware ─────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Model selection ───────────────────────────────────────────────────────────
USE_MARQO      = True
MARQO_MODEL_ID = "marqo/marqo-fashionSigLIP"
MODEL_TAG      = "marqo" if USE_MARQO else "fclip"
EMB_FILE       = WORK_DIR / f"product_embeddings_{MODEL_TAG}.npy"
IDS_FILE       = WORK_DIR / f"product_ids_{MODEL_TAG}.json"

# ── Hyper-params ─────────────────────────────────────────────────────────────
TOP_K            = 15
EMBED_BATCH      = 32  # Lower batch size for Colab (8GB T4 shared VRAM)
DOWNLOAD_WORKERS = 8   # Fewer workers for Colab
IMG_SIZE         = 224
DOWNLOAD_TIMEOUT = 15  # Slightly higher for stability
K_PER_SEGMENT    = 50
AGGREGATION      = "sum"
ENABLE_TEXT_RERANK  = True
TEXT_RERANK_ALPHA   = 0.25
TEXT_RERANK_TOP_N   = 30

print("\n✓ Configuration loaded.")

## Cell 3: Data Loading

In [ ]:
print("Loading CSV files from:", DATA_DIR)
assert DATA_DIR.exists(), f"Data dir not found: {DATA_DIR}\nExpected CSVs in MyDrive/GCED/hackudc/data/"

bundles_df = pd.read_csv(DATA_DIR / "bundles_dataset.csv")
products_df = pd.read_csv(DATA_DIR / "product_dataset.csv")
train_df = pd.read_csv(DATA_DIR / "bundles_product_match_train.csv")
test_df = pd.read_csv(DATA_DIR / "bundles_product_match_test.csv")

print("=== Dataset Statistics ===")
print(f"Bundles total  : {len(bundles_df):,}")
print(f"Products total : {len(products_df):,}")
print(f"Train pairs    : {len(train_df):,}")
print(f"Test bundles   : {len(test_df['bundle_asset_id'].unique()):,}")

# Ground-truth lookup
train_gt: dict[str, set] = (
    train_df.dropna(subset=["product_asset_id"])
    .groupby("bundle_asset_id")["product_asset_id"]
    .apply(set)
    .to_dict()
)

counts = [len(v) for v in train_gt.values()]
print(f"\nTrain bundles with GT  : {len(train_gt):,}")
print(f"Avg products/bundle    : {np.mean(counts):.2f}")
print(f"Max products/bundle    : {max(counts)}")
print(f"Min products/bundle    : {min(counts)}")

# URL & description lookups
bundle_url:   dict[str, str] = dict(zip(bundles_df["bundle_asset_id"], bundles_df["bundle_image_url"]))
product_url:  dict[str, str] = dict(zip(products_df["product_asset_id"], products_df["product_image_url"]))
product_desc: dict[str, str] = dict(zip(products_df["product_asset_id"], products_df["product_description"].fillna("")))

# Category distribution
if "product_description" in products_df.columns:
    cat_counts = products_df["product_description"].value_counts().head(10)
    print("\nTop-10 product categories:")
    print(cat_counts.to_string())

test_bundle_ids = test_df["bundle_asset_id"].dropna().unique().tolist()
print(f"\nTest bundle IDs: {len(test_bundle_ids)}")

## Cell 4: Image Download + Preview

In [ ]:
def download_image(asset_id: str, url: str, out_dir: Path, timeout: int = DOWNLOAD_TIMEOUT) -> bool:
    """Download image if not already cached."""
    out_path = out_dir / f"{asset_id}.jpg"
    if out_path.exists():
        return True
    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        out_path.write_bytes(r.content)
        return True
    except Exception:
        return False

def batch_download(id_url_pairs: list[tuple[str, str]], out_dir: Path, desc: str = "Downloading") -> list[str]:
    """Parallel download with ThreadPoolExecutor."""
    ok_ids: list[str] = []
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        futures = {pool.submit(download_image, aid, url, out_dir): aid for aid, url in id_url_pairs}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=desc):
            aid = futures[fut]
            if fut.result():
                ok_ids.append(aid)
    return ok_ids

def load_image(asset_id: str, img_dir: Path) -> Image.Image | None:
    """Load a cached image as RGB PIL image."""
    p = img_dir / f"{asset_id}.jpg"
    if not p.exists():
        return None
    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None

# Download products & bundles
print("Downloading products…")
product_pairs = [(pid, url) for pid, url in product_url.items()]
valid_product_ids = batch_download(product_pairs, PROD_DIR, desc="Products")
print(f"\nProducts downloaded: {len(valid_product_ids):,} / {len(product_pairs):,} "
      f"({100*len(valid_product_ids)/len(product_pairs):.1f}%)")

print("\nDownloading bundles…")
bundle_pairs = [(bid, url) for bid, url in bundle_url.items()]
valid_bundle_ids = batch_download(bundle_pairs, BUND_DIR, desc="Bundles ")
print(f"Bundles downloaded : {len(valid_bundle_ids):,} / {len(bundle_pairs):,} "
      f"({100*len(valid_bundle_ids)/len(bundle_pairs):.1f}%)")

valid_product_set = set(valid_product_ids)
valid_bundle_set  = set(valid_bundle_ids)

def show_images(asset_ids: list[str], img_dir: Path, title: str = "", n_cols: int = 5):
    """Matplotlib grid preview."""
    imgs = [(aid, load_image(aid, img_dir)) for aid in asset_ids[:n_cols * 3]]
    imgs = [(aid, img) for aid, img in imgs if img is not None]
    if not imgs:
        print("No images to show.")
        return
    n_cols = min(n_cols, len(imgs))
    n_rows = math.ceil(len(imgs) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
    axes = np.array(axes).flatten()
    for ax, (aid, img) in zip(axes, imgs):
        ax.imshow(img)
        ax.set_title(aid[:12], fontsize=7)
        ax.axis("off")
    for ax in axes[len(imgs):]:
        ax.axis("off")
    fig.suptitle(title, fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(WORK_DIR / f"preview_{title.replace(' ','_')}.png", bbox_inches="tight", dpi=80)
    plt.show()
    print(f"Saved: {WORK_DIR}/preview_{title.replace(' ','_')}.png")

show_images(valid_product_ids[:15], PROD_DIR, title="Product Sample")
show_images(valid_bundle_ids[:5],   BUND_DIR, title="Bundle Sample")

## Cell 5: Embedding Models (FashionCLIP + Marqo)

In [ ]:
from fashion_clip.fashion_clip import FashionCLIP
import fashion_clip.fashion_clip as _fc_module
import open_clip

# ── FashionCLIP ─────────────────────────────────────────────────────────────
# Compatibility patch for transformers>=4.40
def _patched_load_model(self, name, device=None, auth_token=None):
    from transformers import CLIPModel, CLIPProcessor
    name = _fc_module._MODELS.get(name, name)
    token_kwargs = {"token": auth_token} if auth_token else {}
    model = CLIPModel.from_pretrained(name, **token_kwargs)
    processor = CLIPProcessor.from_pretrained(name, **token_kwargs)
    hash_val = _fc_module._model_processor_hash(name, model, processor)
    return model, processor, hash_val

FashionCLIP._load_model = _patched_load_model

print("Loading FashionCLIP …")
fclip = FashionCLIP("fashion-clip")
if DEVICE == "cuda":
    fclip.model = fclip.model.to(DEVICE)
fclip.model.eval()
print("✓ FashionCLIP loaded.")

def _clip_encode_images_raw(imgs: list[Image.Image]) -> np.ndarray:
    """FashionCLIP image encoding."""
    inputs = fclip.preprocess(images=imgs, return_tensors="pt", padding=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    pixel_inputs = {"pixel_values": inputs["pixel_values"]}
    with torch.no_grad():
        out = fclip.model.get_image_features(**pixel_inputs)
    if not isinstance(out, torch.Tensor):
        out = out.image_embeds if hasattr(out, "image_embeds") else out.last_hidden_state[:, 0]
    return out.cpu().float().numpy()

def _clip_encode_texts_raw(texts: list[str]) -> np.ndarray:
    """FashionCLIP text encoding."""
    inputs = fclip.preprocess(text=texts, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    text_inputs = {k: v for k, v in inputs.items() if k in ("input_ids", "attention_mask")}
    with torch.no_grad():
        out = fclip.model.get_text_features(**text_inputs)
    if not isinstance(out, torch.Tensor):
        out = out.text_embeds if hasattr(out, "text_embeds") else out.last_hidden_state[:, 0]
    return out.cpu().float().numpy()

def _l2_norm(arr: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(arr, axis=1, keepdims=True).clip(min=1e-8)
    return arr / norms

# ── Marqo-FashionSigLIP via open_clip ───────────────────────────────────────
if USE_MARQO:
    print(f"Loading Marqo-FashionSigLIP …")
    marqo_model, _, marqo_preprocess = open_clip.create_model_and_transforms(
        f"hf-hub:{MARQO_MODEL_ID}"
    )
    marqo_tokenizer = open_clip.get_tokenizer(f"hf-hub:{MARQO_MODEL_ID}")
    marqo_model.to(DEVICE).eval()
    print("✓ Marqo-FashionSigLIP loaded.")

    # Free FashionCLIP VRAM
    del fclip
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print("✓ FashionCLIP unloaded to save VRAM.")

def _marqo_encode_images_raw(imgs: list) -> np.ndarray:
    """Marqo image encoding."""
    tensors = torch.stack([marqo_preprocess(img) for img in imgs]).to(DEVICE)
    with torch.no_grad():
        out = marqo_model.encode_image(tensors)
    return out.cpu().float().numpy()

def _marqo_encode_texts_raw(texts: list[str]) -> np.ndarray:
    """Marqo text encoding."""
    tokens = marqo_tokenizer(texts).to(DEVICE)
    with torch.no_grad():
        out = marqo_model.encode_text(tokens)
    return out.cpu().float().numpy()

# ── Unified dispatch ─────────────────────────────────────────────────────────
def _encode_images_raw(imgs: list) -> np.ndarray:
    return _marqo_encode_images_raw(imgs) if USE_MARQO else _clip_encode_images_raw(imgs)

def _encode_texts_raw(texts: list[str]) -> np.ndarray:
    return _marqo_encode_texts_raw(texts) if USE_MARQO else _clip_encode_texts_raw(texts)

def _embed_images(image_paths: list, batch_size: int = EMBED_BATCH) -> np.ndarray:
    """Embed images with active model."""
    all_embs: list[np.ndarray] = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc="Embedding", leave=False):
        batch_paths = image_paths[i : i + batch_size]
        imgs: list = []
        for p in batch_paths:
            try:
                imgs.append(Image.open(p).convert("RGB"))
            except Exception:
                imgs.append(Image.new("RGB", (IMG_SIZE, IMG_SIZE)))
        embs = _l2_norm(_encode_images_raw(imgs))
        all_embs.append(embs)
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(all_embs, axis=0)

print("✓ All encoding helpers ready.")

## Cell 6: Product Embeddings + FAISS Index

In [ ]:
def _ensure_valid_sets():
    """Rebuild valid sets from disk if needed."""
    global valid_product_set, valid_product_ids, valid_bundle_set, valid_bundle_ids
    prod_on_disk = {p.stem for p in PROD_DIR.glob("*.jpg")}
    bund_on_disk = {p.stem for p in BUND_DIR.glob("*.jpg")}
    if not valid_product_set and prod_on_disk:
        valid_product_set = prod_on_disk
        valid_product_ids = list(prod_on_disk)
        print(f"Rebuilt valid_product_set from disk: {len(valid_product_set):,}")
    if not valid_bundle_set and bund_on_disk:
        valid_bundle_set = bund_on_disk
        valid_bundle_ids = list(bund_on_disk)
        print(f"Rebuilt valid_bundle_set from disk: {len(valid_bundle_set):,}")

_ensure_valid_sets()

if EMB_FILE.exists() and IDS_FILE.exists():
    print("Loading cached product embeddings …")
    product_embeddings = np.load(EMB_FILE)
    with open(IDS_FILE) as f:
        indexed_product_ids: list[str] = json.load(f)
    print(f"  ✓ Loaded {product_embeddings.shape}")
else:
    print("Computing product embeddings (first run, ~15–30 min on Colab T4) …")
    ordered_ids  = [pid for pid in valid_product_ids if pid in valid_product_set]
    ordered_paths = [PROD_DIR / f"{pid}.jpg" for pid in ordered_ids]
    product_embeddings = _embed_images(ordered_paths, batch_size=EMBED_BATCH)
    indexed_product_ids = ordered_ids
    np.save(EMB_FILE, product_embeddings)
    with open(IDS_FILE, "w") as f:
        json.dump(indexed_product_ids, f)
    print(f"  ✓ Saved embeddings: {product_embeddings.shape}")

# Reverse lookup
idx_to_pid: dict[int, str] = {i: pid for i, pid in enumerate(indexed_product_ids)}

# Build FAISS index
DIM = product_embeddings.shape[1]  # 512
print(f"\nBuilding FAISS IndexFlatIP (dim={DIM}, n={len(indexed_product_ids):,}) …")
index_cpu = faiss.IndexFlatIP(DIM)
index_cpu.add(product_embeddings)

if DEVICE == "cuda":
    try:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index_cpu)
        print("  ✓ Transferred index to GPU.")
    except Exception as e:
        print(f"  [WARN] GPU index failed ({e}), using CPU.")
        index = index_cpu
else:
    index = index_cpu
    print("  ✓ Using CPU index.")

print(f"  FAISS ntotal = {index.ntotal:,}")
if index.ntotal < 1_000:
    raise RuntimeError(f"Only {index.ntotal} products indexed — check downloads!")
elif index.ntotal < 20_000:
    print(f"  [WARN] Only {index.ntotal:,} products (expected ~27K). Downloads may have failed.")

## Cell 7: Baseline Pipeline

In [ ]:
def predict_baseline(bundle_ids: list[str], k: int = TOP_K) -> dict[str, list[str]]:
    """Whole-image baseline retrieval."""
    predictions: dict[str, list[str]] = {}
    for bid in tqdm(bundle_ids, desc="Baseline predict"):
        if bid not in valid_bundle_set:
            continue
        img = load_image(bid, BUND_DIR)
        if img is None:
            continue
        emb = _l2_norm(_encode_images_raw([img]))
        scores, indices = index.search(emb, k)
        top_pids = [idx_to_pid[int(i)] for i in indices[0] if int(i) in idx_to_pid]
        predictions[bid] = top_pids[:k]
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return predictions

print("✓ predict_baseline() defined.")

## Cell 8: SegFormer Segmentation

In [ ]:
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import torch.nn.functional as F

print("Loading SegFormer B2 …")
SEG_MODEL_ID = "mattmdjaga/segformer_b2_clothes"
seg_processor = SegformerImageProcessor.from_pretrained(SEG_MODEL_ID)
seg_model = SegformerForSemanticSegmentation.from_pretrained(SEG_MODEL_ID)
seg_model.to(DEVICE).eval()
print("✓ SegFormer loaded.")

ATR_LABELS = {
    0: "Background", 1: "Hat", 2: "Hair", 3: "Sunglasses", 4: "Upper-clothes",
    5: "Skirt", 6: "Pants", 7: "Dress", 8: "Belt", 9: "Left-shoe", 10: "Right-shoe",
    11: "Face", 12: "Left-leg", 13: "Right-leg", 14: "Left-arm", 15: "Right-arm",
    16: "Bag", 17: "Scarf",
}

SEGMENT_GROUPS: dict[str, list[int]] = {
    "upper_body": [4],
    "lower_body":  [5, 6],
    "dress":       [7],
    "shoes":       [9, 10],
    "bag":         [16],
    "hat":         [1],
    "scarf_belt":  [8, 17],
}

SEGMENT_TO_CATEGORIES: dict[str, set[str]] = {
    "upper_body": {"T-SHIRT", "SHIRT", "SWEATER", "WIND-JACKET", "TOPS AND OTHERS", "BLAZER", "SWEATSHIRT", "BABY T-SHIRT", "POLO SHIRT"},
    "lower_body": {"TROUSERS", "SKIRT", "BERMUDA", "BABY TROUSERS", "SHORTS", "LEGGINGS", "JEANS"},
    "dress":      {"DRESS", "OVERALL", "JUMPSUIT"},
    "shoes":      {"SHOE", "SHOES", "SNEAKER", "SNEAKERS", "BOOT", "BOOTS", "SANDAL", "SANDALS", "LOAFER"},
    "bag":        {"HAND BAG-RUCKSACK", "HANDBAG", "BAG", "BACKPACK", "PURSE", "CLUTCH"},
    "hat":        {"HAT", "CAP", "BEANIE"},
    "scarf_belt": {"SCARF", "BELT", "TIE"},
}

ATR_COLORS = plt.cm.get_cmap("tab20", 18)


def visualise_segmentation(pil_img: Image.Image, seg_map: np.ndarray, bundle_id: str = ""):
    """Save a colourmap visualisation of the segmentation alongside the original."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(pil_img)
    ax1.set_title("Original", fontsize=11)
    ax1.axis("off")

    cmap = ListedColormap([ATR_COLORS(i)[:3] for i in range(18)])
    im = ax2.imshow(seg_map, cmap=cmap, vmin=0, vmax=17)
    ax2.set_title("Segmentation", fontsize=11)
    ax2.axis("off")

    patches = [mpatches.Patch(color=ATR_COLORS(i)[:3], label=ATR_LABELS[i]) for i in range(18)]
    ax2.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=7)
    plt.tight_layout()
    plt.savefig(WORK_DIR / f"seg_{bundle_id[:10]}.png", bbox_inches="tight", dpi=80)
    plt.show()

@torch.no_grad()
def segment_image(pil_img: Image.Image) -> np.ndarray:
    """Run SegFormer on image."""
    inputs = seg_processor(images=pil_img, return_tensors="pt").to(DEVICE)
    logits = seg_model(**inputs).logits
    orig_h, orig_w = pil_img.height, pil_img.width
    upsampled = F.interpolate(logits, size=(orig_h, orig_w), mode="bilinear", align_corners=False)
    seg_map = upsampled.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.int32)
    return seg_map

def extract_segment_crops(pil_img: Image.Image, seg_map: np.ndarray, min_frac: float = 0.05, padding: float = 0.05) -> dict[str, Image.Image]:
    """Extract bounding-box crops for each segment."""
    W, H = pil_img.size
    total_pixels = H * W
    crops: dict[str, Image.Image] = {}

    for seg_name, label_ids in SEGMENT_GROUPS.items():
        mask = np.isin(seg_map, label_ids)
        if mask.sum() < min_frac * total_pixels:
            continue

        rows = np.where(mask.any(axis=1))[0]
        cols = np.where(mask.any(axis=0))[0]
        y_min, y_max = int(rows.min()), int(rows.max())
        x_min, x_max = int(cols.min()), int(cols.max())

        pad_y = int((y_max - y_min + 1) * padding)
        pad_x = int((x_max - x_min + 1) * padding)
        y_min = max(0, y_min - pad_y)
        y_max = min(H - 1, y_max + pad_y)
        x_min = max(0, x_min - pad_x)
        x_max = min(W - 1, x_max + pad_x)

        crop = pil_img.crop((x_min, y_min, x_max + 1, y_max + 1))
        if crop.width > 5 and crop.height > 5:
            crops[seg_name] = crop

    return crops

# Demo
demo_bid = next((bid for bid in test_bundle_ids if bid in valid_bundle_set), None)
if demo_bid:
    demo_img = load_image(demo_bid, BUND_DIR)
    if demo_img:
        demo_seg = segment_image(demo_img)
        visualise_segmentation(demo_img, demo_seg, bundle_id=demo_bid)
        demo_crops = extract_segment_crops(demo_img, demo_seg)
        print(f"✓ Demo bundle segments: {list(demo_crops.keys())}")

        # Show crops
        if demo_crops:
            n = len(demo_crops)
            fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
            if n == 1:
                axes = [axes]
            for ax, (name, crop) in zip(axes, demo_crops.items()):
                ax.imshow(crop)
                ax.set_title(name, fontsize=9)
                ax.axis("off")
            plt.tight_layout()
            plt.savefig(WORK_DIR / "demo_crops.png", bbox_inches="tight", dpi=80)
            plt.show()

## Cell 9: Text Re-ranking + Improved Pipeline

In [ ]:
def text_rerank(bundle_id: str, candidate_pids: list[str], crop_embs: np.ndarray, alpha: float = TEXT_RERANK_ALPHA) -> list[str]:
    """Linearly combine image + text similarity."""
    if not candidate_pids:
        return candidate_pids

    descriptions = [product_desc.get(pid, "") for pid in candidate_pids]
    non_empty    = [(i, d) for i, d in enumerate(descriptions) if d.strip()]

    if not non_empty:
        return candidate_pids

    indices_ne, descs_ne = zip(*non_empty)
    text_embs = _l2_norm(_encode_texts_raw(list(descs_ne)))

    sim = crop_embs @ text_embs.T
    text_scores_ne = sim.max(axis=0)

    img_rank_score = np.array([1.0 - i / len(candidate_pids) for i in range(len(candidate_pids))])

    text_score_full = np.zeros(len(candidate_pids))
    for list_idx, orig_idx in enumerate(indices_ne):
        text_score_full[orig_idx] = float(text_scores_ne[list_idx])

    combined = (1 - alpha) * img_rank_score + alpha * text_score_full
    order    = np.argsort(combined)[::-1]
    return [candidate_pids[j] for j in order]

print("✓ text_rerank() defined.")

def predict_segmented(bundle_ids: list[str], k: int = TOP_K, fallback_to_whole: bool = True) -> dict[str, list[str]]:
    """Per-segment retrieval with category filtering + sum aggregation + text re-ranking."""
    predictions: dict[str, list[str]] = {}

    for bid in tqdm(bundle_ids, desc="Segmented predict"):
        if bid not in valid_bundle_set:
            continue

        img = load_image(bid, BUND_DIR)
        if img is None:
            continue

        # Segmentation
        try:
            seg_map = segment_image(img)
            crops = extract_segment_crops(img, seg_map)
        except Exception as e:
            print(f"  [WARN] Segmentation failed for {bid}: {e}")
            crops = {}

        # Embedding
        if crops:
            crop_list = list(crops.values())
            seg_names = list(crops.keys())
        else:
            crop_list = [img]
            seg_names = ["whole"]

        embs_norm = _l2_norm(_encode_images_raw(crop_list))

        # FAISS query per segment
        scores_map: dict[str, float] = {}

        scores_arr, indices_arr = index.search(embs_norm, K_PER_SEGMENT)

        for seg_name, seg_scores, seg_indices in zip(seg_names, scores_arr, indices_arr):
            allowed = {c.upper() for c in SEGMENT_TO_CATEGORIES.get(seg_name, set())}
            for sc, idx_val in zip(seg_scores, seg_indices):
                pid = idx_to_pid.get(int(idx_val))
                if pid is None:
                    continue
                if allowed and product_desc.get(pid, "").strip().upper() not in allowed:
                    continue
                scores_map[pid] = scores_map.get(pid, 0.0) + float(sc)

        # Text re-ranking
        ranked_ext = sorted(scores_map.items(), key=lambda x: x[1], reverse=True)
        top_pids = [pid for pid, _ in ranked_ext[:TEXT_RERANK_TOP_N]]
        if ENABLE_TEXT_RERANK and top_pids:
            top_pids = text_rerank(bid, top_pids, embs_norm, alpha=TEXT_RERANK_ALPHA)

        predictions[bid] = top_pids[:k]

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    return predictions

_ensure_valid_sets()
print("Running segmented pipeline …")
segmented_preds = predict_segmented(test_bundle_ids, k=TOP_K)
print(f"✓ Segmented predictions for {len(segmented_preds)} bundles.")

## Cell 10: Validation (Recall@15)

In [ ]:
def recall_at_k(predictions: dict[str, list[str]], ground_truth: dict[str, set], k: int = TOP_K) -> float:
    """Macro-averaged Recall@k."""
    total_recall = 0.0
    n = 0
    for bid, gt_set in ground_truth.items():
        if not gt_set:
            continue
        preds = predictions.get(bid, [])[:k]
        hit = len(set(preds) & gt_set)
        total_recall += hit / len(gt_set)
        n += 1
    return total_recall / n if n > 0 else 0.0

train_bundle_ids_with_gt = list(train_gt.keys())
print("Running baseline on train bundles for validation …")
baseline_train_preds = predict_baseline(train_bundle_ids_with_gt)
print("Running segmented on train bundles for validation …")
seg_train_preds = predict_segmented(train_bundle_ids_with_gt)

recall_base = recall_at_k(baseline_train_preds, train_gt)
recall_seg  = recall_at_k(seg_train_preds, train_gt)

print(f"\n{'='*40}")
print(f"Recall@{TOP_K}  Baseline  : {recall_base:.4f}")
print(f"Recall@{TOP_K}  Segmented : {recall_seg:.4f}")
print(f"Delta                  : {recall_seg - recall_base:+.4f}")
print(f"{'='*40}")

# Per-complexity breakdown
complexity_bins = {
    "easy (1-2)":   [bid for bid, gt in train_gt.items() if 1 <= len(gt) <= 2],
    "medium (3-5)": [bid for bid, gt in train_gt.items() if 3 <= len(gt) <= 5],
    "hard (6+)":    [bid for bid, gt in train_gt.items() if len(gt) >= 6],
}

print("\nPer-complexity Recall@15:")
base_vals, seg_vals, labels = [], [], []
for label, bids in complexity_bins.items():
    sub_gt = {b: train_gt[b] for b in bids if b in train_gt}
    r_base = recall_at_k(baseline_train_preds, sub_gt)
    r_seg  = recall_at_k(seg_train_preds, sub_gt)
    print(f"  {label:16s}  Baseline={r_base:.4f}  Segmented={r_seg:.4f}  n={len(bids)}")
    base_vals.append(r_base)
    seg_vals.append(r_seg)
    labels.append(label)

# Bar chart
x = np.arange(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, base_vals, w, label="Baseline",  color="steelblue")
ax.bar(x + w/2, seg_vals,  w, label="Segmented", color="coral")
ax.set_ylabel("Recall@15")
ax.set_title("Recall@15 by Bundle Complexity")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
plt.tight_layout()
plt.savefig(WORK_DIR / "recall_comparison.png", bbox_inches="tight", dpi=100)
plt.show()

## Cell 11: Generate Submission

In [ ]:
baseline_preds = predict_baseline(test_bundle_ids, k=TOP_K)

use_segmented = recall_seg >= recall_base
method_name   = "segmented" if use_segmented else "baseline"
final_preds   = segmented_preds if use_segmented else baseline_preds
print(f"Selected method: {method_name} (Recall@{TOP_K} = {recall_seg if use_segmented else recall_base:.4f})")

from collections import Counter

product_freq = Counter(pid for gt_set in train_gt.values() for pid in gt_set if pid in valid_product_set)
fallback_pids = [pid for pid, _ in product_freq.most_common(TOP_K)]
print(f"Fallback products: {fallback_pids[:5]} …")

rows: list[dict] = []
missing_count = 0
for bid in test_bundle_ids:
    preds = final_preds.get(bid, [])
    if not preds:
        preds = fallback_pids
        missing_count += 1
    if len(preds) < TOP_K:
        for fp in fallback_pids:
            if fp not in preds:
                preds.append(fp)
            if len(preds) >= TOP_K:
                break
    preds = preds[:TOP_K]
    for pid in preds:
        rows.append({"bundle_asset_id": bid, "product_asset_id": pid})

submission_df = pd.DataFrame(rows)
submission_df.to_csv(SUBM_FILE, index=False)
print(f"\n✓ Submission saved to {SUBM_FILE}")
print(f"  Rows: {len(submission_df):,}")
print(f"  Fallback bundles: {missing_count}")

# Sanity checks
print("\n" + "="*40)
covered = set(submission_df["bundle_asset_id"].unique())
missing = set(test_bundle_ids) - covered
assert len(missing) == 0, f"Missing bundles: {missing}"
print(f"✓ All {len(test_bundle_ids)} test bundles present.")

max_preds = submission_df.groupby("bundle_asset_id")["product_asset_id"].count().max()
assert max_preds <= TOP_K
print(f"✓ Max predictions per bundle: {max_preds}")

predicted_pids = set(submission_df["product_asset_id"])
invalid_pids   = predicted_pids - valid_product_set
if invalid_pids:
    print(f"[WARN] {len(invalid_pids)} invalid product IDs — removing.")
    submission_df = submission_df[~submission_df["product_asset_id"].isin(invalid_pids)]
    submission_df.to_csv(SUBM_FILE, index=False)
else:
    print(f"✓ All {len(predicted_pids):,} product IDs valid.")

print("="*40)
print("\nSubmission preview (first 20 rows):")
print(submission_df.head(20).to_string(index=False))

## Cell 12: Text Re-ranking Demo

In [ ]:
# text_rerank() is defined in Cell 9 — demo only here.
print("Demonstrating text re-ranking …")

demo_rerank_bids = [bid for bid in train_bundle_ids_with_gt[:3] if bid in seg_train_preds]

for bid in demo_rerank_bids:
    img = load_image(bid, BUND_DIR)
    if img is None:
        continue

    try:
        seg_map = segment_image(img)
        crops   = extract_segment_crops(img, seg_map)
    except Exception:
        crops = {}

    crop_imgs = list(crops.values()) if crops else [img]

    ce = _l2_norm(_encode_images_raw(crop_imgs))

    before = seg_train_preds.get(bid, [])[:TOP_K]
    after  = text_rerank(bid, before[:], ce)

    gt = train_gt.get(bid, set())
    hit_before = len(set(before) & gt)
    hit_after  = len(set(after) & gt)
    print(f"  {bid}: hits before={hit_before}, after text-rerank={hit_after} (gt={len(gt)})")

## Summary

In [ ]:
print("\n" + "=" * 60)
print("HACKUDC 2026 — SOLUTION SUMMARY (Colab)")
print("=" * 60)
print(f"Products indexed         : {index.ntotal:,}")
print(f"Embedding dimension      : {DIM}")
print(f"FAISS backend            : {'GPU' if DEVICE == 'cuda' else 'CPU'}")
print(f"Segmentation model       : SegFormer B2 (ATR 17 classes)")
print(f"Retrieval model          : {'Marqo-FashionSigLIP' if USE_MARQO else 'FashionCLIP'} (512-dim)")
print(f"Baseline Recall@{TOP_K}    : {recall_base:.4f}")
print(f"Segmented Recall@{TOP_K}   : {recall_seg:.4f}")
print(f"Method selected          : {method_name}")
print(f"Submission file          : {SUBM_FILE}")
print(f"Submission rows          : {len(submission_df):,}")
print("=" * 60)